In [ ]:
# =============================================================================
# CONCENTRIC FRINGE PATTERN DETECTION FOR SAR POINT SCATTERERS
# =============================================================================

from scipy.ndimage import sobel, label, center_of_mass
from scipy.optimize import minimize
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.feature import corner_peaks
from scipy.signal import correlate2d
import warnings
warnings.filterwarnings('ignore')

class SAR_FringeDetector:
    """
    Class for detecting concentric fringe patterns around point scatterers in SAR imagery.
    Implements multiple algorithms for robust detection under noisy conditions.
    """
    
    def __init__(self):
        self.detected_centers = []
        self.confidence_scores = []
    
    def radial_gradient_analysis(self, image, candidate_centers=None, search_radius=20):
        """
        Detect fringe centers by analyzing radial gradient patterns.
        
        Args:
            image: 2D numpy array (amplitude or phase)
            candidate_centers: List of (y,x) tuples for candidate positions
            search_radius: Maximum search radius around candidates
            
        Returns:
            List of detected centers with confidence scores
        """
        if candidate_centers is None:
            # Generate grid of candidate points
            h, w = image.shape
            y_candidates = np.arange(search_radius, h-search_radius, search_radius//2)
            x_candidates = np.arange(search_radius, w-search_radius, search_radius//2)
            candidate_centers = [(y, x) for y in y_candidates for x in x_candidates]
        
        detected_centers = []
        
        for cy, cx in candidate_centers:
            # Extract local region
            y1, y2 = max(0, cy-search_radius), min(image.shape[0], cy+search_radius)
            x1, x2 = max(0, cx-search_radius), min(image.shape[1], cx+search_radius)
            
            local_region = image[y1:y2, x1:x2]
            local_cy, local_cx = cy - y1, cx - x1
            
            # Compute radial gradient score
            score = self._compute_radial_gradient_score(local_region, local_cy, local_cx)
            
            if score > 0.3:  # Threshold for detection
                detected_centers.append(((cy, cx), score))
        
        # Sort by confidence and return top candidates
        detected_centers.sort(key=lambda x: x[1], reverse=True)
        return detected_centers[:10]  # Return top 10 candidates
    
    def _compute_radial_gradient_score(self, image, cy, cx):
        """
        Compute how well the image exhibits radial gradient pattern around center.
        """
        h, w = image.shape
        if cy <= 1 or cx <= 1 or cy >= h-2 or cx >= w-2:
            return 0.0
        
        # Create radial distance map
        y, x = np.ogrid[:h, :w]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)
        
        # Sample along different radii
        angles = np.linspace(0, 2*np.pi, 16, endpoint=False)
        radial_profiles = []
        
        max_radius = min(cy, cx, h-cy, w-cx, 15)
        if max_radius < 5:
            return 0.0
        
        for angle in angles:
            # Sample along this radial direction
            radii = np.arange(1, max_radius)
            y_coords = cy + radii * np.sin(angle)
            x_coords = cx + radii * np.cos(angle)
            
            # Ensure coordinates are within bounds
            valid_mask = ((y_coords >= 0) & (y_coords < h) & 
                         (x_coords >= 0) & (x_coords < w))
            
            if np.sum(valid_mask) < 3:
                continue
                
            y_coords = y_coords[valid_mask].astype(int)
            x_coords = x_coords[valid_mask].astype(int)
            
            profile = image[y_coords, x_coords]
            radial_profiles.append(profile)
        
        if len(radial_profiles) < 8:
            return 0.0
        
        # Analyze oscillatory behavior in radial profiles
        oscillation_scores = []
        for profile in radial_profiles:
            if len(profile) > 4:
                # Compute gradient to detect transitions
                grad = np.gradient(profile)
                # Count sign changes (oscillations)
                sign_changes = np.sum(np.diff(np.sign(grad)) != 0)
                oscillation_scores.append(sign_changes / len(profile))
        
        if not oscillation_scores:
            return 0.0
            
        return np.mean(oscillation_scores)
    
    def template_matching_approach(self, image, template_sizes=[5, 7, 9, 11]):
        """
        Use template matching with synthetic concentric ring patterns.
        
        Args:
            image: Input SAR image (amplitude or phase)
            template_sizes: List of template sizes to try
            
        Returns:
            List of detected centers with correlation scores
        """
        detected_centers = []
        
        for size in template_sizes:
            # Create concentric ring template
            template = self._create_ring_template(size)
            
            # Perform normalized cross-correlation
            correlation = correlate2d(image, template, mode='valid')
            
            # Find local maxima in correlation
            from scipy.ndimage import maximum_filter
            local_maxima = maximum_filter(correlation, size=5) == correlation
            
            # Threshold and extract peaks
            threshold = np.mean(correlation) + 2 * np.std(correlation)
            peaks = np.where((correlation > threshold) & local_maxima)
            
            for y, x in zip(peaks[0], peaks[1]):
                # Adjust coordinates for template offset
                center_y = y + size // 2
                center_x = x + size // 2
                score = correlation[y, x]
                detected_centers.append(((center_y, center_x), score))
        
        # Remove duplicates and sort by score
        detected_centers = self._remove_duplicate_detections(detected_centers, min_distance=10)
        detected_centers.sort(key=lambda x: x[1], reverse=True)
        
        return detected_centers[:15]
    
    def _create_ring_template(self, size):
        """
        Create a synthetic concentric ring template.
        """
        center = size // 2
        y, x = np.ogrid[:size, :size]
        r = np.sqrt((y - center)**2 + (x - center)**2)
        
        # Create alternating rings
        template = np.sin(2 * np.pi * r * 3 / size)
        
        # Apply Gaussian envelope to focus on center
        envelope = np.exp(-r**2 / (2 * (size/4)**2))
        template *= envelope
        
        return template
    
    def phase_coherence_method(self, complex_image, window_size=15):
        """
        Detect fringe centers using phase coherence analysis.
        
        Args:
            complex_image: Complex SAR image (Re + 1j*Im)
            window_size: Size of analysis window
            
        Returns:
            List of detected centers based on phase coherence
        """
        phase = np.angle(complex_image)
        amplitude = np.abs(complex_image)
        
        h, w = phase.shape
        coherence_map = np.zeros((h, w))
        
        half_window = window_size // 2
        
        for y in range(half_window, h - half_window):
            for x in range(half_window, w - half_window):
                # Extract local window
                local_phase = phase[y-half_window:y+half_window+1, 
                                  x-half_window:x+half_window+1]
                local_amp = amplitude[y-half_window:y+half_window+1, 
                                    x-half_window:x+half_window+1]
                
                # Compute phase coherence score
                coherence_map[y, x] = self._compute_phase_coherence(local_phase, local_amp, 
                                                                   half_window, half_window)
        
        # Find peaks in coherence map
        from scipy.ndimage import maximum_filter
        local_maxima = maximum_filter(coherence_map, size=7) == coherence_map
        
        threshold = np.mean(coherence_map) + 1.5 * np.std(coherence_map)
        peaks = np.where((coherence_map > threshold) & local_maxima)
        
        detected_centers = []
        for y, x in zip(peaks[0], peaks[1]):
            score = coherence_map[y, x]
            detected_centers.append(((y, x), score))
        
        detected_centers.sort(key=lambda x: x[1], reverse=True)
        return detected_centers[:10]
    
    def _compute_phase_coherence(self, phase_window, amp_window, cy, cx):
        """
        Compute phase coherence score for concentric pattern detection.
        """
        h, w = phase_window.shape
        
        # Create radial distance map
        y, x = np.ogrid[:h, :w]
        r = np.sqrt((y - cy)**2 + (x - cx)**2)
        
        # Group pixels by radial distance
        max_radius = int(np.max(r))
        coherence_scores = []
        
        for radius in range(1, max_radius):
            mask = (r >= radius - 0.5) & (r < radius + 0.5)
            if np.sum(mask) < 3:
                continue
                
            phase_samples = phase_window[mask]
            amp_samples = amp_window[mask]
            
            # Weight by amplitude
            weights = amp_samples / np.sum(amp_samples)
            
            # Compute circular variance (measure of phase coherence)
            complex_sum = np.sum(weights * np.exp(1j * phase_samples))
            coherence = np.abs(complex_sum)
            coherence_scores.append(coherence)
        
        if not coherence_scores:
            return 0.0
            
        # Look for alternating pattern in coherence
        coherence_scores = np.array(coherence_scores)
        if len(coherence_scores) > 3:
            # Compute variance in coherence (indicates fringe pattern)
            return np.var(coherence_scores)
        else:
            return np.mean(coherence_scores)
    
    def _remove_duplicate_detections(self, detections, min_distance=10):
        """
        Remove duplicate detections that are too close to each other.
        """
        if not detections:
            return []
            
        # Sort by score (highest first)
        detections.sort(key=lambda x: x[1], reverse=True)
        
        filtered_detections = []
        
        for detection in detections:
            center, score = detection
            
            # Check if this detection is too close to any existing detection
            too_close = False
            for existing_center, _ in filtered_detections:
                distance = np.sqrt((center[0] - existing_center[0])**2 + 
                                 (center[1] - existing_center[1])**2)
                if distance < min_distance:
                    too_close = True
                    break
            
            if not too_close:
                filtered_detections.append(detection)
        
        return filtered_detections
    
    def detect_fringe_centers(self, image, complex_image=None, method='all'):
        """
        Main method to detect fringe centers using multiple approaches.
        
        Args:
            image: Amplitude or phase image
            complex_image: Complex SAR image (optional)
            method: 'radial', 'template', 'phase', or 'all'
            
        Returns:
            Dictionary with results from different methods
        """
        results = {}
        
        if method in ['radial', 'all']:
            print("Running radial gradient analysis...")
            results['radial'] = self.radial_gradient_analysis(image)
        
        if method in ['template', 'all']:
            print("Running template matching...")
            results['template'] = self.template_matching_approach(image)
        
        if method in ['phase', 'all'] and complex_image is not None:
            print("Running phase coherence analysis...")
            results['phase'] = self.phase_coherence_method(complex_image)
        
        return results

# Initialize the detector
fringe_detector = SAR_FringeDetector()

print("✅ SAR Fringe Pattern Detection algorithms loaded successfully!")
print("Available methods:")
print("  - radial_gradient_analysis(): Analyzes radial gradient patterns")
print("  - template_matching_approach(): Uses synthetic ring templates")
print("  - phase_coherence_method(): Analyzes phase coherence")
print("  - detect_fringe_centers(): Main detection method")

In [ ]:
# =============================================================================
# ADVANCED NOISE-ROBUST FRINGE DETECTION TECHNIQUES
# =============================================================================

class AdvancedFringeDetector:
    """
    Advanced techniques for detecting concentric fringes under heavy noise conditions.
    """
    
    def __init__(self):
        self.detection_threshold = 0.5
    
    def multi_scale_detection(self, image, scales=[1, 2, 4]):
        """
        Detect fringes at multiple scales to handle different fringe sizes.

        Args:
            image: Input SAR image
            scales: List of downsampling factors
            
        Returns:
            Combined detection results from all scales
        """
        all_detections = []
        
        for scale in scales:
            if scale == 1:
                scaled_image = image
            else:
                # Downsample image
                from skimage.transform import resize
                new_shape = (image.shape[0] // scale, image.shape[1] // scale)
                scaled_image = resize(image, new_shape, anti_aliasing=True)
            
            # Apply template matching at this scale
            detector = SAR_FringeDetector()
            detections = detector.template_matching_approach(scaled_image)
            
            # Scale coordinates back to original image
            for (y, x), score in detections:
                original_y = int(y * scale)
                original_x = int(x * scale)
                all_detections.append(((original_y, original_x), score / scale))
        
        # Remove duplicates and return top candidates
        detector = SAR_FringeDetector()
        filtered_detections = detector._remove_duplicate_detections(all_detections, min_distance=15)
        filtered_detections.sort(key=lambda x: x[1], reverse=True)
        
        return filtered_detections[:10]
    
    def ensemble_detection(self, image, complex_image=None):
        """
        Combine multiple detection methods for robust results.

        Args:
            image: Input SAR image (amplitude or phase)
            complex_image: Complex SAR image (optional)
            
        Returns:
            Ensemble detection results with confidence scores
        """
        all_detections = []
        
        # 1. Basic detectors
        basic_detector = SAR_FringeDetector()
        basic_results = basic_detector.detect_fringe_centers(image, complex_image)
        
        # Weight and combine basic results
        method_weights = {'radial': 1.0, 'template': 1.2, 'phase': 0.8}
        
        for method, detections in basic_results.items():
            weight = method_weights.get(method, 1.0)
            for (y, x), score in detections:
                all_detections.append(((y, x), score * weight, method))
        
        # 2. Multi-scale detection
        ms_detections = self.multi_scale_detection(image)
        for (y, x), score in ms_detections:
            all_detections.append(((y, x), score * 1.1, 'multiscale'))
        
        # 3. Group nearby detections and compute ensemble score
        ensemble_detections = self._compute_ensemble_scores(all_detections)
        
        return ensemble_detections
    
    def _compute_ensemble_scores(self, all_detections, cluster_radius=15):
        """
        Compute ensemble scores by clustering nearby detections.
        """
        if not all_detections:
            return []
        
        # Group detections by proximity
        clusters = []
        used = set()
        
        for i, (center_i, score_i, method_i) in enumerate(all_detections):
            if i in used:
                continue
                
            cluster = [(center_i, score_i, method_i)]
            used.add(i)
            
            # Find nearby detections
            for j, (center_j, score_j, method_j) in enumerate(all_detections):
                if j in used:
                    continue
                    
                distance = np.sqrt((center_i[0] - center_j[0])**2 + 
                                 (center_i[1] - center_j[1])**2)
                
                if distance < cluster_radius:
                    cluster.append((center_j, score_j, method_j))
                    used.add(j)
            
            clusters.append(cluster)
        
        # Compute ensemble scores for each cluster
        ensemble_results = []
        
        for cluster in clusters:
            if len(cluster) == 1:
                # Single detection
                center, score, method = cluster[0]
                ensemble_results.append((center, score))
            else:
                # Multiple detections - compute weighted average
                centers = np.array([c[0] for c in cluster])
                scores = np.array([c[1] for c in cluster])
                methods = [c[2] for c in cluster]
                
                # Weight by score
                weights = scores / np.sum(scores)
                
                # Weighted average center
                avg_center = np.average(centers, axis=0, weights=weights)
                avg_center = (int(avg_center[0]), int(avg_center[1]))
                
                # Ensemble score: mean score * diversity bonus
                mean_score = np.mean(scores)
                diversity_bonus = len(set(methods)) / len(cluster)  # Bonus for method diversity
                ensemble_score = mean_score * (1 + diversity_bonus)
                
                ensemble_results.append((avg_center, ensemble_score))
        
        # Sort by ensemble score
        ensemble_results.sort(key=lambda x: x[1], reverse=True)
        return ensemble_results

# Initialize advanced detector
advanced_detector = AdvancedFringeDetector()

print("✨ Advanced SAR Fringe Detection techniques loaded!")
print("Available advanced methods:")
print("  - multi_scale_detection(): Detect at multiple image scales")
print("  - ensemble_detection(): Combine multiple methods")

In [ ]:
# Imports and Initial Setup
import os 
import argparse # Kept for reference, but parameters will be set manually
import configparser
import pandas as pd
import torch
import pytorch_lightning as pl


from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objs as go
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from pynas.core.population import Population
from datasets.RawVessels.loader import RawVesselsDataModule, RawVesselsDataset

import numpy as np
import cv2
# Additional imports for statistical threshold calculation
from scipy.optimize import minimize
from matplotlib import pyplot as plt
from scipy.ndimage import gaussian_filter, median_filter
from scipy.signal import hilbert
from skimage import io
from skimage.restoration import denoise_tv_chambolle
from skimage.feature import blob_log, peak_local_max

# Pandas display option
pd.set_option('display.max_colwidth', None)
# PyTorch Lightning seed and precision
pl.seed_everything(seed=42, workers=True) # Example seed, change as needed
torch.set_float32_matmul_precision("medium")

# ----- Load the dataset ------
root_dir_datamodule = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/data/TASI/DataSAR_real_refined'
# Ensure the root directory exists
if not Path(root_dir_datamodule).exists():
    raise FileNotFoundError(f"Root directory {root_dir_datamodule} does not exist.")
# Load image and mask paths
root_dir_datamodule = Path(root_dir_datamodule)
if not (root_dir_datamodule / 'inputs').exists() or not (root_dir_datamodule / 'masks').exists():
    raise FileNotFoundError(f"Expected directories 'inputs' and 'masks' not found in {root_dir_datamodule}.")

image_paths = [x for x in (root_dir_datamodule / 'inputs').glob('*.pkl')]
mask_paths = [x for x in (root_dir_datamodule / 'masks').glob('*.pkl')]


loader = RawVesselsDataset(image_paths, 
                           mask_paths, 
                           transform=None)


# Try loading and inspecting a sample
rand_idx = 5  # Change this index to load different samples
sample = loader[rand_idx]
img, mask = sample 


Re, Im = img  # Real and Imaginary parts of the complex image
# Make Amplitude and Phase
amp = np.abs(Re + 1j * Im)  # Amplitude
phase = np.angle(Re + 1j * Im)  # Phase

# =============================================================================
# APPLY FRINGE DETECTION TO THE LOADED SAR DATA
# =============================================================================

# Create complex image from real and imaginary parts
# Check if Re and Im have channel dimension and extract first channel if needed
if len(Re.shape) == 3:
    # If shape is (2, H, W), we need to take the appropriate channels
    complex_img = Re[0] + 1j * Im[0]  # Take first channel
else:
    complex_img = Re + 1j * Im

print(f"Complex image shape: {complex_img.shape}")
print(f"Re shape: {Re.shape}, Im shape: {Im.shape}")

# Test all detection methods
print("🎯 Detecting concentric fringe patterns in SAR data...")
print(f"Image shape: {amp.shape}")
print(f"Phase range: [{np.min(phase):.3f}, {np.max(phase):.3f}]")
print(f"Amplitude range: [{np.min(amp):.3f}, {np.max(amp):.3f}]")

# Run detection on amplitude image
results_amp = fringe_detector.detect_fringe_centers(amp, complex_img, method='all')

# Run detection on phase image
results_phase = fringe_detector.detect_fringe_centers(phase, complex_img, method='all')

print("\n📊 Detection Results Summary:")
print("\nAmplitude-based detection:")
for method, detections in results_amp.items():
    print(f"  {method}: {len(detections)} candidates")
    if detections:
        top_3 = detections[:3]
        for i, ((y, x), score) in enumerate(top_3):
            print(f"    #{i+1}: ({y:3d}, {x:3d}) score={score:.3f}")

print("\nPhase-based detection:")
for method, detections in results_phase.items():
    print(f"  {method}: {len(detections)} candidates")
    if detections:
        top_3 = detections[:3]
        for i, ((y, x), score) in enumerate(top_3):
            print(f"    #{i+1}: ({y:3d}, {x:3d}) score={score:.3f}")


In [ ]:
# =============================================================================
# APPLY ADVANCED DETECTION AND COMPREHENSIVE ANALYSIS
# =============================================================================

# Run ensemble detection on amplitude image
print("🎆 Running advanced ensemble detection on amplitude image...")
ensemble_results_amp = advanced_detector.ensemble_detection(amp, complex_img)

print(f"\n🏆 Ensemble detection found {len(ensemble_results_amp)} high-confidence candidates:")
for i, ((y, x), score) in enumerate(ensemble_results_amp[:5]):
    print(f"  #{i+1}: Center=({y:3d}, {x:3d}), Confidence={score:.3f}")

# Run ensemble detection on phase image
print("\n🎆 Running advanced ensemble detection on phase image...")
ensemble_results_phase = advanced_detector.ensemble_detection(phase_smoothed, complex_img)

print(f"\n🏆 Ensemble detection found {len(ensemble_results_phase)} high-confidence candidates:")
for i, ((y, x), score) in enumerate(ensemble_results_phase[:5]):
    print(f"  #{i+1}: Center=({y:3d}, {x:3d}), Confidence={score:.3f}")

# Visualize ensemble results
def plot_ensemble_results(image, ensemble_results, title="Ensemble Detection", max_show=5):
    """
    Plot ensemble detection results with confidence indicators.
    """
    fig, ax = plt.subplots(1, 1, figsize=(10, 8), dpi=120)
    
    # Display image
    if 'phase' in title.lower():
        ax.imshow(image, cmap='inferno', vmin=vmin, vmax=vmax)
    else:
        ax.imshow(image, cmap='gray')
    
    # Color map for confidence levels
    colors = plt.cm.Reds(np.linspace(0.4, 1.0, max_show))
    
    # Plot detections with confidence-based styling
    for i, ((y, x), confidence) in enumerate(ensemble_results[:max_show]):
        color = colors[i]
        
        # Main detection point
        ax.plot(x, y, 'o', color=color, markersize=12, 
               markeredgecolor='white', markeredgewidth=2,
               label=f'#{i+1} (conf={confidence:.2f})')
        
        # Confidence rings
        ring_alpha = 0.3 + 0.4 * (confidence / max(c for _, c in ensemble_results[:max_show]))
        for radius in [8, 16, 24]:
            circle = plt.Circle((x, y), radius, fill=False, 
                              color=color, alpha=ring_alpha, linewidth=2)
            ax.add_patch(circle)
        
        # Add confidence text
        ax.text(x + 5, y - 5, f'{confidence:.2f}', 
               fontsize=10, color='white', weight='bold',
               bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.8))
    
    ax.set_title(f'{title}\nTop {min(max_show, len(ensemble_results))} High-Confidence Detections')
    ax.axis('off')
    ax.legend(loc='upper right', fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Plot ensemble results
print("\n🖼️ Visualizing ensemble detection results:")

if ensemble_results_amp:
    plot_ensemble_results(amp, ensemble_results_amp, "Amplitude - Ensemble Detection")

if ensemble_results_phase:
    plot_ensemble_results(phase_smoothed, ensemble_results_phase, "Phase - Ensemble Detection")

# Create comprehensive comparison
print("\n🏆 Method Comparison Summary:")
print("\nAmplitude-based methods:")
all_amp_methods = {
    'Radial_Gradient': results_amp.get('radial', []),
    'Template_Matching': results_amp.get('template', []),
    'Ensemble': ensemble_results_amp
}

for method, detections in all_amp_methods.items():
    if detections:
        top_score = detections[0][1] if detections else 0
        print(f"  {method:20s}: {len(detections):2d} detections, top score: {top_score:.3f}")
    else:
        print(f"  {method:20s}: No detections")

print("\nPhase-based methods:")
all_phase_methods = {
    'Phase_Coherence': results_phase.get('phase', []),
    'Phase_Ensemble': ensemble_results_phase
}

for method, detections in all_phase_methods.items():
    if detections:
        top_score = detections[0][1] if detections else 0
        print(f"  {method:20s}: {len(detections):2d} detections, top score: {top_score:.3f}")
    else:
        print(f"  {method:20s}: No detections")

# Final summary visualization
def create_summary_visualization(amp_img, phase_img, amp_results, phase_results):
    """
    Create a comprehensive summary visualization of all detection results.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), dpi=120)
    
    # Amplitude with all detections
    axes[0,0].imshow(amp_img, cmap='gray')
    axes[0,0].set_title('Amplitude - All Detection Methods')
    
    # Plot different methods with different markers
    markers = ['o', 's', '^', 'D', 'v']
    colors = ['red', 'yellow', 'cyan', 'lime', 'magenta']
    
    method_idx = 0
    for method, detections in amp_results.items():
        if detections and method_idx < len(markers):
            for i, ((y, x), score) in enumerate(detections[:3]):  # Show top 3 per method
                axes[0,0].plot(x, y, markers[method_idx], color=colors[method_idx], 
                              markersize=8, markeredgecolor='white', markeredgewidth=1,
                              label=f'{method}' if i == 0 else "")
            method_idx += 1
    
    axes[0,0].legend(fontsize=8)
    axes[0,0].axis('off')
    
    # Phase with detections
    axes[0,1].imshow(phase_img, cmap='inferno', vmin=vmin, vmax=vmax)
    axes[0,1].set_title('Phase - Detection Methods')
    
    method_idx = 0
    for method, detections in phase_results.items():
        if detections and method_idx < len(markers):
            for i, ((y, x), score) in enumerate(detections[:3]):
                axes[0,1].plot(x, y, markers[method_idx], color=colors[method_idx], 
                              markersize=8, markeredgecolor='white', markeredgewidth=1,
                              label=f'{method}' if i == 0 else "")
            method_idx += 1
    
    axes[0,1].legend(fontsize=8)
    axes[0,1].axis('off')
    
    # Detection count comparison
    method_names = list(amp_results.keys()) + list(phase_results.keys())
    detection_counts = ([len(d) for d in amp_results.values()] + 
                       [len(d) for d in phase_results.values()])
    
    bars = axes[1,0].bar(range(len(method_names)), detection_counts, 
                        color=['skyblue']*len(amp_results) + ['lightcoral']*len(phase_results))
    axes[1,0].set_title('Number of Detections per Method')
    axes[1,0].set_xticks(range(len(method_names)))
    axes[1,0].set_xticklabels(method_names, rotation=45, ha='right')
    axes[1,0].set_ylabel('Number of Detections')
    axes[1,0].grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, count in zip(bars, detection_counts):
        if count > 0:
            axes[1,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                          str(count), ha='center', va='bottom')
    
    # Confidence score comparison (top detection from each method)
    top_scores = []
    score_methods = []
    
    for method, detections in amp_results.items():
        if detections:
            top_scores.append(detections[0][1])
            score_methods.append(f'{method} (Amp)')
    
    for method, detections in phase_results.items():
        if detections:
            top_scores.append(detections[0][1])
            score_methods.append(f'{method} (Phase)')
    
    if top_scores:
        bars = axes[1,1].bar(range(len(score_methods)), top_scores,
                           color=['skyblue']*len(amp_results) + ['lightcoral']*len(phase_results))
        axes[1,1].set_title('Top Confidence Scores per Method')
        axes[1,1].set_xticks(range(len(score_methods)))
        axes[1,1].set_xticklabels(score_methods, rotation=45, ha='right')
        axes[1,1].set_ylabel('Confidence Score')
        axes[1,1].grid(True, alpha=0.3)
        
        # Add value labels
        for bar, score in zip(bars, top_scores):
            axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                          f'{score:.2f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.suptitle('SAR Concentric Fringe Detection - Comprehensive Results', 
                 fontsize=16, y=1.02)
    plt.show()

# Generate final summary
print("\n🖼️ Generating comprehensive results visualization...")
create_summary_visualization(amp, phase_smoothed, all_amp_methods, all_phase_methods)

# Final recommendations
print("\n🎆 RECOMMENDATIONS FOR SAR FRINGE DETECTION:")
print("\n1. 🏆 BEST METHODS:")
print("   - Ensemble Detection: Combines multiple approaches for robustness")
print("   - Template Matching: Good for regular fringe patterns")
print("   - Radial Gradient: Effective for noisy conditions")

print("\n2. 🛠️ PREPROCESSING RECOMMENDATIONS:")
print("   - Apply Gaussian smoothing to phase images (sigma=2-4)")
print("   - Use amplitude images for initial detection")
print("   - Consider multi-scale analysis for varying fringe sizes")

print("\n3. 📊 VALIDATION STRATEGIES:")
print("   - Check radial intensity profiles for oscillatory behavior")
print("   - Verify symmetry around detected centers")
print("   - Use phase coherence as additional validation")

print("\n4. ⚠️ NOISE HANDLING:")
print("   - Ensemble methods are most robust to noise")
print("   - Consider denoising before detection in very noisy images")
print("   - Use confidence thresholding to filter weak detections")

print("\n✅ Analysis complete! The implemented algorithms provide a comprehensive")
print("   framework for detecting concentric fringe patterns in SAR imagery.")
print("\n📁 Use these methods by calling:")
print("   - fringe_detector.detect_fringe_centers(image, complex_image, method='all')")
print("   - advanced_detector.ensemble_detection(image, complex_image)")
print("   - advanced_detector.multi_scale_detection(image)")

In [ ]:
# =============================================================================
# VISUALIZE DETECTION RESULTS
# =============================================================================

def plot_detection_results(image, detections_dict, title_prefix="", max_detections=5):
    """
    Plot detection results overlaid on the original image.
    """
    n_methods = len(detections_dict)
    if n_methods == 0:
        return
    
    fig, axes = plt.subplots(1, n_methods, figsize=(5*n_methods, 5), dpi=120)
    if n_methods == 1:
        axes = [axes]
    
    colors = ['red', 'yellow', 'cyan', 'lime', 'magenta']
    
    for idx, (method, detections) in enumerate(detections_dict.items()):
        ax = axes[idx]
        
        # Display base image
        if 'phase' in title_prefix.lower():
            im = ax.imshow(image, cmap='inferno', vmin=vmin, vmax=vmax)
        else:
            im = ax.imshow(image, cmap='gray')
        
        # Overlay detections
        for i, ((y, x), score) in enumerate(detections[:max_detections]):
            color = colors[i % len(colors)]
            
            # Draw detection point
            ax.plot(x, y, 'o', color=color, markersize=8, markeredgecolor='white', 
                   markeredgewidth=2, label=f'#{i+1} (s={score:.2f})')
            
            # Draw concentric circles to highlight fringe pattern
            for radius in [5, 10, 15, 20]:
                if y-radius >= 0 and y+radius < image.shape[0] and \
                   x-radius >= 0 and x+radius < image.shape[1]:
                    circle = plt.Circle((x, y), radius, fill=False, 
                                      color=color, alpha=0.6, linewidth=1.5)
                    ax.add_patch(circle)
        
        ax.set_title(f'{title_prefix} - {method.capitalize()}\n{len(detections)} detections')
        ax.axis('off')
        
        if detections:
            ax.legend(loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()

# Plot results for amplitude-based detection
print("🖼️ Amplitude-based fringe detection results:")
plot_detection_results(amp, results_amp, "Amplitude", max_detections=3)

# Plot results for phase-based detection
print("\n🖼️ Phase-based fringe detection results:")
plot_detection_results(phase_smoothed, results_phase, "Phase (Smoothed)", max_detections=3)

In [ ]:
# =============================================================================
# DETAILED ANALYSIS OF DETECTED FRINGE PATTERNS
# =============================================================================

def analyze_fringe_pattern(image, center_y, center_x, max_radius=25, title="Fringe Analysis"):
    """
    Perform detailed analysis of the fringe pattern around a detected center.
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 10), dpi=120)
    
    # Extract region around center
    y1, y2 = max(0, center_y-max_radius), min(image.shape[0], center_y+max_radius)
    x1, x2 = max(0, center_x-max_radius), min(image.shape[1], center_x+max_radius)
    
    local_region = image[y1:y2, x1:x2]
    local_cy, local_cx = center_y - y1, center_x - x1
    
    # 1. Original region with center marked
    axes[0,0].imshow(local_region, cmap='gray')
    axes[0,0].plot(local_cx, local_cy, 'r+', markersize=15, markeredgewidth=3)
    axes[0,0].set_title('Local Region')
    axes[0,0].axis('off')
    
    # 2. Radial distance map
    h, w = local_region.shape
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((y - local_cy)**2 + (x - local_cx)**2)
    
    axes[0,1].imshow(r, cmap='viridis')
    axes[0,1].plot(local_cx, local_cy, 'r+', markersize=15, markeredgewidth=3)
    axes[0,1].set_title('Radial Distance Map')
    axes[0,1].axis('off')
    
    # 3. Radial profiles
    angles = np.linspace(0, 2*np.pi, 8, endpoint=False)
    radii = np.arange(1, min(max_radius, local_cy, local_cx, h-local_cy, w-local_cx))
    
    axes[0,2].set_title('Radial Intensity Profiles')
    for i, angle in enumerate(angles):
        y_coords = local_cy + radii * np.sin(angle)
        x_coords = local_cx + radii * np.cos(angle)
        
        # Ensure coordinates are within bounds
        valid_mask = ((y_coords >= 0) & (y_coords < h) & 
                     (x_coords >= 0) & (x_coords < w))
        
        if np.sum(valid_mask) > 3:
            y_coords = y_coords[valid_mask].astype(int)
            x_coords = x_coords[valid_mask].astype(int)
            valid_radii = radii[valid_mask]
            
            profile = local_region[y_coords, x_coords]
            axes[0,2].plot(valid_radii, profile, alpha=0.7, 
                          label=f'{angle*180/np.pi:.0f}°')
    
    axes[0,2].set_xlabel('Radius (pixels)')
    axes[0,2].set_ylabel('Intensity')
    axes[0,2].legend(fontsize=8)
    axes[0,2].grid(True, alpha=0.3)
    
    # 4. Azimuthally averaged profile
    max_r = int(np.min([max_radius, local_cy, local_cx, h-local_cy, w-local_cx]))
    avg_profile = []
    std_profile = []
    
    for radius in range(1, max_r):
        mask = (r >= radius - 0.5) & (r < radius + 0.5)
        if np.sum(mask) > 0:
            values = local_region[mask]
            avg_profile.append(np.mean(values))
            std_profile.append(np.std(values))
        else:
            avg_profile.append(0)
            std_profile.append(0)
    
    radii_avg = np.arange(1, len(avg_profile) + 1)
    avg_profile = np.array(avg_profile)
    std_profile = np.array(std_profile)
    
    axes[1,0].plot(radii_avg, avg_profile, 'b-', linewidth=2, label='Mean')
    axes[1,0].fill_between(radii_avg, avg_profile - std_profile, 
                          avg_profile + std_profile, alpha=0.3, label='±1σ')
    axes[1,0].set_title('Azimuthally Averaged Profile')
    axes[1,0].set_xlabel('Radius (pixels)')
    axes[1,0].set_ylabel('Intensity')
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)
    
    # 5. Gradient magnitude
    grad_y = sobel(local_region, axis=0)
    grad_x = sobel(local_region, axis=1)
    grad_mag = np.sqrt(grad_y**2 + grad_x**2)
    
    axes[1,1].imshow(grad_mag, cmap='hot')
    axes[1,1].plot(local_cx, local_cy, 'w+', markersize=15, markeredgewidth=3)
    axes[1,1].set_title('Gradient Magnitude')
    axes[1,1].axis('off')
    
    # 6. Oscillation analysis
    if len(avg_profile) > 4:
        # Compute derivatives to find oscillations
        grad_profile = np.gradient(avg_profile)
        second_grad = np.gradient(grad_profile)
        
        axes[1,2].plot(radii_avg, grad_profile, 'g-', label='1st Derivative')
        axes[1,2].plot(radii_avg, second_grad, 'r-', label='2nd Derivative')
        axes[1,2].axhline(y=0, color='k', linestyle='--', alpha=0.5)
        axes[1,2].set_title('Profile Derivatives')
        axes[1,2].set_xlabel('Radius (pixels)')
        axes[1,2].legend()
        axes[1,2].grid(True, alpha=0.3)
        
        # Count zero crossings (oscillations)
        zero_crossings = np.sum(np.diff(np.sign(grad_profile)) != 0)
        axes[1,2].text(0.05, 0.95, f'Zero crossings: {zero_crossings}', 
                      transform=axes[1,2].transAxes, 
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.suptitle(f'{title} - Center: ({center_y}, {center_x})', fontsize=14)
    plt.tight_layout()
    plt.show()

# Analyze the top detections from each method
print("🔍 Detailed analysis of top detections:\n")

# Analyze top amplitude-based detection
if results_amp.get('radial') and len(results_amp['radial']) > 0:
    top_center = results_amp['radial'][0][0]
    print(f"Analyzing top radial detection in amplitude: {top_center}")
    analyze_fringe_pattern(amp, top_center[0], top_center[1], 
                          title="Amplitude - Radial Method")

# Analyze top template matching detection
if results_amp.get('template') and len(results_amp['template']) > 0:
    top_center = results_amp['template'][0][0]
    print(f"\nAnalyzing top template detection in amplitude: {top_center}")
    analyze_fringe_pattern(amp, top_center[0], top_center[1], 
                          title="Amplitude - Template Method")

# Analyze top phase coherence detection
if results_phase.get('phase') and len(results_phase['phase']) > 0:
    top_center = results_phase['phase'][0][0]
    print(f"\nAnalyzing top phase coherence detection: {top_center}")
    analyze_fringe_pattern(phase_smoothed, top_center[0], top_center[1], 
                          title="Phase - Coherence Method")